In [19]:
import os
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter  


In [20]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [35]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 66be99ca-b0dc-4ec6-8a74-ae54284ff4d4)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./config_sentence_transformers.json
Retrying in 1s [Retry 1/5].


In [22]:

llm

ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001DCED379A70>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001DCED379CD0>, root_client=<openai.OpenAI object at 0x000001DCEABD2F10>, root_async_client=<openai.AsyncOpenAI object at 0x000001DCEABD3130>, model_name='gpt-4.1', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [23]:
class AgentState(TypedDict):
    question: str
    documents: List[Document]
    answer: str
    needs_retrieval: bool

In [31]:
class AgentState(TypedDict):
    question: str
    documents: List[Document]
    answer: str
    needs_retrieval: bool

In [32]:
# Sample documents for demonstration
sample_texts = [
    "LangGraph is a library for building stateful, multi-actor applications with LLMs. It extends LangChain with the ability to coordinate multiple chains across multiple steps of computation in a cyclic manner.",
    "RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text generation. It retrieves relevant documents and uses them to provide context for generating more accurate responses.",
    "Vector databases store high-dimensional vectors and enable efficient similarity search. They are commonly used in RAG systems to find relevant documents based on semantic similarity.",
    "Agentic systems are AI systems that can take actions, make decisions, and interact with their environment autonomously. They often use planning and reasoning capabilities."
]

documents=[Document(page_content=text) for text in sample_texts]

##create vector store
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(k=3)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: 9wjsVgAR**********************************************************************************************************************************xVvn. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [32]:
# First, install the required package:
# pip install sentence-transformers

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document  # ✅ new path


sample_texts = [
    "LangGraph is a library for building stateful, multi-actor applications with LLMs...",
    "RAG (Retrieval-Augmented Generation) is a technique...",
    "Vector databases store high-dimensional vectors...",
    "Agentic systems are AI systems that can take actions..."
]

documents = [Document(page_content=text) for text in sample_texts]

# SWITCH to local embeddings - no API calls, no rate limits!
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# This will work without any rate limits
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Success! Vector store created with local embeddings.")

C:\Users\niico\AppData\Local\Temp\ipykernel_3808\704814976.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: eaf7c45d-f290-4c70-b7f7-e5dbecf744d9)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./README.md
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: e807c90a-cd03-477b-9bf2-cd74133667df)')' thrown while requesting HEAD ht

Success! Vector store created with local embeddings.


In [33]:
import time
from tenacity import retry, stop_after_attempt, wait_exponential
from langchain_openai import OpenAIEmbeddings

# First, check your current embeddings setup
# You're probably using something like:
# embeddings = OpenAIEmbeddings(openai_api_key="your-key")

@retry(
    stop=stop_after_attempt(5),  # Try 5 times
    wait=wait_exponential(multiplier=2, min=10, max=60)  # Longer waits: 10s to 60s
)
def create_vectorstore_with_retry(documents, embeddings):
    try:
        return FAISS.from_documents(documents, embeddings)
    except Exception as e:
        print(f"Attempt failed: {e}")
        time.sleep(30)  # Additional 30-second delay
        raise  # Re-raise for tenacity to handle

# Use it with longer delays
vectorstore = create_vectorstore_with_retry(documents, embeddings)

In [34]:
# Add this to see what embeddings you're using
print(f"Embeddings type: {type(embeddings)}")
print(f"Embeddings config: {embeddings}")

# If it's OpenAIEmbeddings, you'll see rate limits
# Switch to local like this:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Embeddings type: <class 'langchain_community.embeddings.huggingface.HuggingFaceEmbeddings'>
Embeddings config: client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
) model_name='all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} multi_process=False show_progress=False


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 264e0726-ae23-40b3-a2d0-e540c5862b31)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 78a97342-3feb-4faa-8e52-48f80dfbe5ab)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./config_sentence_transformers.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 62d1b501-08ae-4ebb-b19d-d7dcef5b2810)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./README.md
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(hos